# Project : AI Vision: Screen-to-Code Generator Using VLMs
## Shahab Esfandiar

In [1]:
import os
import re
import time
import base64
import traceback
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv()

API_KEY = os.getenv("AVALAI_API_KEY")
BASE_URL = os.getenv("AVALAI_BASE_URL", "https://api.avalai.ir/v1")

if not API_KEY:
    raise ValueError("⚠️ AVALAI_API_KEY not found in .env file.")

# Initialize the OpenAI client for Vision capabilities
client = OpenAI(
    api_key=API_KEY, 
    base_url=BASE_URL,
    max_retries=2
)

In [3]:
# SECTION 2: VISION FRONT-END ENGINEER ENGINE
class VisionFrontendEngineer:
    def __init__(self, image_bytes, image_name, framework="Tailwind CSS"):
        """Initialize the engine with image data, metadata, and the target framework."""
        self.image_bytes = image_bytes
        self.image_name = image_name
        self.framework = framework
        
        # Track token usage to calculate costs later
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.EXCHANGE_RATE = 153000 
        
    def _encode_image(self):
        """Convert raw binary image data into a Base64 encoded string for the API."""
        return base64.b64encode(self.image_bytes).decode('utf-8')

    def _determine_mime_type(self):
        """Dynamically detect the MIME type based on the file extension."""
        ext = self.image_name.split('.')[-1].lower()
        if ext in ['png']: return 'image/png'
        elif ext in ['webp']: return 'image/webp'
        else: return 'image/jpeg' 

    def build_system_prompt(self):
        """
        Construct a strict prompt focused entirely on pixel-perfect replication.
        Removed distracting instructions (like SEO/Hover states) to maximize visual accuracy.
        """
        prompt = (
            "You are an expert Frontend Developer. Your ONLY task is to recreate the provided UI screenshot "
            "with absolute pixel-perfect accuracy.\n\n"
            "CRITICAL CONSTRAINTS:\n"
            "1. Output exactly ONE single valid HTML file containing all styles and structures.\n"
            "2. Do NOT hallucinate, summarize, or skip content. Write the complete code for everything visible in the image, including sidebars, menus, and text.\n"
            "3. Replicate the exact colors, spacing, borders, and layout structure.\n"
            "4. Do NOT wrap your response in markdown blocks (e.g., ```html). Output RAW text only.\n"
            f"5. You MUST strictly use {self.framework} for styling and layout building.\n"
        )
        # Inject framework-specific CDN scripts
        if self.framework == "Tailwind CSS":
            prompt += "Include Tailwind via <script src='[https://cdn.tailwindcss.com](https://cdn.tailwindcss.com)'></script>. You can use arbitrary values (e.g., bg-[#0d1117]) to match exact colors."
        elif self.framework == "Bootstrap 5":
            prompt += "Include Bootstrap via <link href='[https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css](https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css)' rel='stylesheet'>."
            
        return prompt

    def generate_code_with_retry(self, max_retries=3):
        """Execute the Vision API call with robust error handling and timeout limits."""
        base64_image = self._encode_image()
        mime_type = self._determine_mime_type()
        system_instruction = self.build_system_prompt()
        
        # Construct the multi-modal payload containing both the prompt and the base64 image
        payload_messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": [
                {"type": "text", "text": "Recreate this exact UI step-by-step. Pay close attention to the layout grids, sidebar widths, and exact text content."},
                {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{base64_image}", "detail": "high"}}
            ]}
        ]
        
        for attempt in range(max_retries):
            try:
                # Increased temperature to 0.2 to allow the model enough creativity to handle complex, long layouts without truncating.
                # High timeout (180s) because processing a dense image like a GitHub dashboard is computationally expensive.
                response = client.chat.completions.create(
                    model="gpt-4o",
                    messages=payload_messages,
                    temperature=0.2, 
                    timeout=180.0 
                )
                
                # Accumulate token usage
                self.prompt_tokens += response.usage.prompt_tokens
                self.completion_tokens += response.usage.completion_tokens
                
                # Sanitize the output to remove persistent markdown code blocks
                raw_code = response.choices[0].message.content.strip()
                raw_code = re.sub(r'^```(html|xml|vue|javascript|css)?\s*', '', raw_code, flags=re.IGNORECASE)
                raw_code = re.sub(r'\s*```$', '', raw_code)
                return raw_code
                
            except Exception as e:
                # Backoff logic: wait before retrying if the network drops
                if attempt < max_retries - 1:
                    time.sleep(3 ** attempt) 
                else:
                    raise RuntimeError(f"VLM API completely failed after {max_retries} attempts. Error: {str(e)}")

    def save_project(self, code, base_filename):
        """Save the generated HTML string into a physical file within the Outputs directory."""
        output_dir = "Outputs"
        os.makedirs(output_dir, exist_ok=True)
        file_path = os.path.join(output_dir, f"{base_filename}.html")
        
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(code)
        return file_path

    def render_analytics(self):
        """Calculate and display the financial cost of the Vision API call."""
        input_usd = (self.prompt_tokens / 1000) * 0.005
        output_usd = (self.completion_tokens / 1000) * 0.015
        total_usd = input_usd + output_usd
        total_toman = total_usd * self.EXCHANGE_RATE
        
        analytics_dashboard = f"""
        <div style="border: 2px solid #8e44ad; padding: 20px; border-radius: 10px; margin-top: 20px; background-color: #fcf3ff; direction: ltr; text-align: left;">
            <h3 style="color: #8e44ad; margin-top: 0;">👁️ Vision AI Token & Cost Analytics</h3>
            <table style="width: 100%; border-collapse: collapse; background: white;">
                <tr style="background-color: #8e44ad; color: white;"><th style="padding: 10px;">Metric Type</th><th style="padding: 10px;">Token Volume</th></tr>
                <tr><td style="padding: 10px; border-bottom: 1px solid #ddd;">Image + Prompt Tokens</td><td style="padding: 10px; border-bottom: 1px solid #ddd;">{self.prompt_tokens:,}</td></tr>
                <tr><td style="padding: 10px; border-bottom: 1px solid #ddd;">Generated Code Tokens</td><td style="padding: 10px; border-bottom: 1px solid #ddd;">{self.completion_tokens:,}</td></tr>
                <tr style="color: #27ae60; font-weight: bold;"><td style="padding: 10px;">Estimated Cost</td><td style="padding: 10px;">${total_usd:.4f} USD (~{total_toman:,.0f} Toman)</td></tr>
            </table>
        </div>
        """
        display(HTML(analytics_dashboard))

In [4]:
# SECTION 3: VLM INTERACTIVE DASHBOARD

# 1. UI Components Setup
image_uploader = widgets.FileUpload(
    accept='image/*', # Accept all standard image formats
    multiple=False,
    description='📸 Upload UI Screenshot',
    button_style='primary',
    layout=widgets.Layout(width='220px')
)

image_name_label = widgets.HTML("<span style='color: #888; font-size: 14px; margin-left: 15px;'>No image selected</span>")

def on_image_upload(change):
    """Dynamic observer to update the label when an image is selected."""
    if image_uploader.value:
        uploaded_data = image_uploader.value
        name = uploaded_data[0]['name'] if isinstance(uploaded_data, tuple) else list(uploaded_data.keys())[0]
        image_name_label.value = f"<span style='color: #8e44ad; font-weight: bold; font-size: 14px; margin-left: 15px;'>🖼️ Image ready: {name}</span>"

image_uploader.observe(on_image_upload, names='value')
upload_box = widgets.HBox([image_uploader, image_name_label], layout=widgets.Layout(align_items='center', margin='0 0 20px 0'))

framework_selector = widgets.ToggleButtons(
    options=['Vanilla CSS', 'Tailwind CSS', 'Bootstrap 5'],
    value='Tailwind CSS',
    description='🎨 UI Framework:',
    button_style='info',
    style={'description_width': '120px'}
)

generate_btn = widgets.Button(
    description='Generate Code from Image',
    button_style='success',
    icon='code',
    layout=widgets.Layout(width='250px', height='45px', margin='20px 10px 20px 0px')
)

reset_vlm_btn = widgets.Button(
    description='Reset',
    button_style='danger',
    icon='refresh',
    layout=widgets.Layout(width='120px', height='45px', margin='20px 0px')
)

vlm_buttons_row = widgets.HBox([generate_btn, reset_vlm_btn])
vlm_log_monitor = widgets.Output()

# 2. Dynamic CSS Loading Animation Component
spinner_html = """
<div style="display: flex; align-items: center; color: #8e44ad; margin: 15px 0; font-family: sans-serif;">
    <div style="border: 4px solid #f3f3f3; border-top: 4px solid #8e44ad; border-radius: 50%; width: 26px; height: 26px; animation: spin 1s linear infinite; margin-right: 15px;"></div>
    <span style="font-size: 15px;"><b>Analyzing pixels and writing code...</b> <br><small style="color:#666;">(Vision processing usually takes 1 to 3 minutes. Please wait...)</small></span>
    <style>@keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }</style>
</div>
"""
loading_widget = widgets.HTML("")

def reset_vlm_dashboard(b):
    """Clear logs, reset uploader, and hide spinner."""
    vlm_log_monitor.clear_output()
    image_uploader.value = () if isinstance(image_uploader.value, tuple) else {}
    image_name_label.value = "<span style='color: #888; font-size: 14px; margin-left: 15px;'>No image selected</span>"
    loading_widget.value = ""

reset_vlm_btn.on_click(reset_vlm_dashboard)

# 3. Execution Pipeline Logic
def trigger_vlm_pipeline(b):
    with vlm_log_monitor:
        # Always clear the output window upon a new execution run
        clear_output() 
        loading_widget.value = ""
        
        if not image_uploader.value:
            print("⚠️ Error: Please select an image (screenshot or wireframe) first.")
            return
            
        start_time = time.time()
        
        try:
            # Extract binary content safely depending on ipywidgets version
            uploaded_data = image_uploader.value
            file_meta = uploaded_data[0] if isinstance(uploaded_data, tuple) else list(uploaded_data.values())[0]
                
            img_bytes = file_meta['content']
            raw_name = file_meta['name']
            clean_base_name = os.path.splitext(raw_name)[0]
            framework = framework_selector.value
            
            output_title = f"UI_Code_{clean_base_name}_{framework.replace(' ', '')}"
            
            print(f"🚀 Initializing AI Vision Engine for: {raw_name}")
            print(f"⚙️ Target Framework: {framework}")
            print("="*70)
            
            # Display the animated CSS spinner while Python blocks for the API call
            loading_widget.value = spinner_html
            display(loading_widget)
            
            # Execute the core Vision AI engine
            vlm_engine = VisionFrontendEngineer(image_bytes=img_bytes, image_name=raw_name, framework=framework)
            generated_code = vlm_engine.generate_code_with_retry()
            
            # Replace spinner with a success message
            loading_widget.value = "<div style='color: #27ae60; font-weight: bold; margin: 15px 0;'>✅ Code generation successful! Structuring layout...</div>"
            
            # Export the generated HTML payload to the file system
            saved_path = vlm_engine.save_project(generated_code, output_title)
            
            # Calculate total execution duration
            end_time = time.time()
            mins, secs = divmod(end_time - start_time, 60)
            
            print("="*70 + "\n🎉 Project Compiled Successfully!")
            print(f"⏱️ Time Taken: {int(mins)}m {int(secs)}s")
            print(f"🌐 View your website: Open the file -> {saved_path}")
            
            # Print financial metrics table
            vlm_engine.render_analytics()

        except Exception as e:
            loading_widget.value = "" # Hide spinner on error to prevent confusion
            print("\n" + "!"*70)
            print("🛑 CRITICAL SYSTEM ERROR")
            print(f"Details: {str(e)}")
            print("!"*70)
            traceback.print_exc()

# Bind the execution logic to the generate button
generate_btn.on_click(trigger_vlm_pipeline)

# 4. Final Layout Rendering
vlm_ui_card = widgets.VBox([
    widgets.HTML("<h2 style='color: #8e44ad;'>👁️ AI Vision: Screen-to-Code Generator</h2>"),
    widgets.HTML("<p style='color: #666;'>Upload a screenshot or hand-drawn wireframe, choose your CSS framework, and let the AI build your frontend UI.</p><hr>"),
    upload_box,
    widgets.VBox([framework_selector], layout=widgets.Layout(margin='10px 0')),
    vlm_buttons_row,
    vlm_log_monitor
], layout=widgets.Layout(padding='25px', border='1px solid #ddd', border_radius='10px', background_color='#fbf8ff'))

display(vlm_ui_card)